In [1]:
import sys
# Add the desired directory to sys.path
sys.path.append('build/')

import randSVD
from sklearn.utils.extmath import randomized_svd

seed = 7050
tol = 1e-3

## Test matrix

In [6]:
import numpy as np

class TestMatrix:
    def __init__(self, rows, cols, sing_val_decay):
        self.rows = rows
        self.cols = cols
        self.sing_val_decay = sing_val_decay
        
        # Calculate rank
        self.rank = min(rows, cols)
        
        # Generate random U and V matrices
        self.U = np.random.rand(rows, self.rank)
        self.V = np.random.rand(cols, self.rank)
        
        # Orthogonalize U and V using QR decomposition
        self.U, _ = np.linalg.qr(self.U)
        self.V, _ = np.linalg.qr(self.V)
        
        # Set the singular values based on the decay option
        if self.sing_val_decay == "fast":
            self.sing_vals = np.power(0.95, np.arange(self.rank))
        else:
            self.sing_vals = 1/np.log(2 + np.arange(self.rank))

        self.A = (self.U @ np.diag(self.sing_vals) @ self.V.T).astype(np.float64)
        
    def matrixU(self):
        return self.U
    
    def singularValues(self):
        return self.sing_vals
    
    def matrixV(self):
        return self.V
    
    def matrixA(self):
        return self.A
    

## Error metrics

In [7]:
def compute_errors(test_matrix,U,sing_vals,V):
    errors = {}

    rank = sing_vals.size

    errors["rec_error"] = np.linalg.norm(test_matrix.matrixA() - U@np.diag(sing_vals)@V.T, 'fro')
    errors["sing_val_error"] = np.linalg.norm(test_matrix.singularValues()[:rank] - sing_vals)

    errors["l_sing_vect_error"] = max(
        np.minimum(np.linalg.norm(test_matrix.matrixU()[:,:rank]-U,axis=0),
                   np.linalg.norm(test_matrix.matrixU()[:,:rank]+U,axis=0))
                   )    
    errors["r_sing_vect_error"] = max(
        np.minimum(np.linalg.norm(test_matrix.matrixV()[:,:rank]-V,axis=0),
                   np.linalg.norm(test_matrix.matrixV()[:,:rank]+V,axis=0))
                   )
    return errors

## Computations

In [10]:
import pandas as pd
import time  

# Assuming randSVD and TestMatrix are already defined somewhere

test_results = pd.DataFrame(columns=['svd', 'size', 'replica', 'rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error', 'execution_time'])
tested_sizes = [1000, 2000, 3000, 4000, 5000]

rank = 5  # to compare with rbki
n_iter = 10
tol = 1e-10
seed = 7050

n_replicas = 1

rsi = randSVD.RSI(seed, tol)
rbki = randSVD.RBKI(seed, tol)

for test_size in tested_sizes:
    print("Size: ", test_size)
    test_matrix = TestMatrix(test_size, test_size, "slow")

    for i in range(1, n_replicas + 1):
        # RSI computation
        start_time = time.time()
        rsi.compute(test_matrix.matrixA(), rank, n_iter)
        execution_time = time.time() - start_time
        errors_rsi = compute_errors(test_matrix, rsi.matrixU(), rsi.singularValues(), rsi.matrixV())
        test_results = pd.concat([test_results, pd.DataFrame([["rsi", test_size, i, *errors_rsi.values(), execution_time]], 
                                  columns=['svd', 'size', 'replica', 'rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error', 'execution_time'])], 
                                 ignore_index=True)

        # RBKI computation
        start_time = time.time()
        rbki.compute(test_matrix.matrixA(), rank, n_iter)
        execution_time = time.time() - start_time
        errors_rbki = compute_errors(test_matrix, rbki.matrixU(), rbki.singularValues(), rbki.matrixV())
        test_results = pd.concat([test_results, pd.DataFrame([["rbki", test_size, i, *errors_rbki.values(), execution_time]], 
                                  columns=['svd', 'size', 'replica', 'rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error', 'execution_time'])], 
                                 ignore_index=True)
        
        # Scikit-learn computation (randomized SVD)
        start_time = time.time()
        U, s, Vh = randomized_svd(test_matrix.matrixA(),
                                   n_components=rank,
                                   n_oversamples=rank,
                                   n_iter=n_iter,
                                   random_state=0)
        execution_time = time.time() - start_time
        errors_sklearn = compute_errors(test_matrix, U, s, Vh.T)
        test_results = pd.concat([test_results, pd.DataFrame([["scikit-learn", test_size, i, *errors_sklearn.values(), execution_time]], 
                                  columns=['svd', 'size', 'replica', 'rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error', 'execution_time'])], 
                                 ignore_index=True)
        

Size:  1000


/var/folders/7g/9rtz76hn4j5_ws3_z5kszxk40000gn/T/ipykernel_1660/3364377774.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  test_results = pd.concat([test_results, pd.DataFrame([["rsi", test_size, i, *errors_rsi.values(), execution_time]],


Size:  2000
Size:  3000
Size:  4000
Size:  5000


In [11]:
print(test_results)

             svd  size replica  rec_error  sing_val_error  l_sing_vect_error  \
0            rsi  1000       1   5.643737    1.197888e-06       3.137714e-03   
1           rbki  1000       1   5.643737    6.327307e-14       6.620127e-11   
2   scikit-learn  1000       1   5.643737    2.957354e-07       1.481807e-03   
3            rsi  2000       1   7.119274    5.337389e-08       6.156692e-04   
4           rbki  2000       1   7.119274    5.585536e-14       8.027695e-11   
5   scikit-learn  2000       1   7.119274    1.624697e-07       1.101351e-03   
6            rsi  3000       1   8.188794    9.416372e-08       8.278912e-04   
7           rbki  3000       1   8.188794    5.330319e-14       1.202802e-10   
8   scikit-learn  3000       1   8.188794    2.009899e-06       4.195617e-03   
9            rsi  4000       1   9.060263    3.450309e-07       1.605112e-03   
10          rbki  4000       1   9.060263    5.179675e-14       1.833218e-10   
11  scikit-learn  4000       1   9.06026

In [12]:
test_results.to_csv('results/randSVD_comparison.csv', index=False)